# Agent de Preprocessing Automatise — Demonstration Complete

**Projet :** Agent IA de preprocessing Data Science base sur GPT avec validation humaine.

**Objectif :** A partir d'un fichier CSV et d'une colonne cible, produire automatiquement un dataset nettoye (train/test) pret pour la modelisation, accompagne d'un rapport qualite detaille.

---

### Architecture du pipeline

Le pipeline est un **graphe LangGraph** a 11 noeuds avec 3 points d'interruption (human-in-the-loop).

```
START -> analyze -> split_data -> [INTERRUPT domain_review]
      -> propose_transformations -> [INTERRUPT transformation_review]
      -> execute_transformations -> propose_outlier_treatment -> [INTERRUPT outlier_review]
      -> execute_outlier_treatment -> apply_scaling -> generate_report -> END
```

**Principes de conception :**
- **Separation rule/LLM** : les decisions deterministes (>50% NaN -> drop, cardinalite >50 -> frequency encoding) sont prises par des regles. Le LLM n'intervient que sur les cas ambigus.
- **Train-only fit** : tous les encodeurs, scalers et bornes outliers sont ajustes sur le train uniquement. Le test est transforme avec les memes parametres.
- **Tracabilite** : chaque decision est accompagnee d'une source (`rule`/`llm`), d'un niveau de confiance, et d'une justification.
- **Validation Pydantic** : les reponses LLM sont validees par des schemas stricts (17 actions autorisees, retry automatique si invalide).

### Plan du notebook

| Section | Contenu |
|---------|--------|
| 1. Setup | Imports, initialisation du pipeline |
| 2. Titanic | Dataset de reference — walkthrough complet etape par etape |
| 3. Tips | Dataset propre — regression, peu de transformations |
| 4. Diamonds | Grand dataset (54k lignes) — haute cardinalite, performance |
| 5. Penguins | Multi-classe avec NaN — classification 3 classes |
| 6. Cas extremes | Dataset synthetique — colonnes 100% NaN, constantes, IDs |
| 7. Adult Census | Grand dataset (32k), valeurs sentinelles "?", mix categorique |
| 8. Mpg | Petit dataset, "?" dans horsepower, cast_numeric, regression |
| 9. Wine Quality | Tout numerique, pas de NaN, teste scaling et outliers |
| 10. Robustesse | Target invalide, gestion d'erreur |
| 11. Synthese | Tableau comparatif, analyse qualite, limites |

## 1. Setup

In [1]:
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path

from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from src.agents.pipeline import build_pipeline

c:\Users\abdel\GenAI project\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
def run_pipeline_auto(df_path: str, target: str, thread_id: str = "test"):
    """Execute le pipeline complet avec auto-approve a chaque interrupt.

    Capture toutes les informations intermediaires (domain, transformations,
    leakage, outliers, quality score) pour analyse post-run.
    """
    checkpointer = MemorySaver()
    pipeline = build_pipeline(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": thread_id}}

    results = {"df_path": df_path, "target": target, "errors": []}
    start = time.time()

    try:
        pipeline.invoke({"df_path": df_path, "target": target}, config=config)

        snapshot = pipeline.get_state(config)
        domain_interrupt = snapshot.tasks[0].interrupts[0].value
        results["domain_context"] = domain_interrupt.get("domain_context", {})
        results["semantic_anomalies"] = domain_interrupt.get("semantic_anomalies", [])

        pipeline.invoke(Command(resume="approve"), config=config)

        snapshot = pipeline.get_state(config)
        transform_interrupt = snapshot.tasks[0].interrupts[0].value
        results["transform_proposals"] = transform_interrupt.get("proposals", {})
        results["leakage_alerts"] = transform_interrupt.get("leakage_alerts", [])
        results["correlation_alerts"] = transform_interrupt.get("correlation_alerts", [])

        pipeline.invoke(Command(resume="approve"), config=config)

        # Gestion de l'interrupt optionnel leakage_warning
        snapshot = pipeline.get_state(config)
        if snapshot.tasks and snapshot.tasks[0].interrupts:
            interrupt_value = snapshot.tasks[0].interrupts[0].value
            if interrupt_value.get("type") == "leakage_warning":
                results["leakage_warning"] = True
                pipeline.invoke(Command(resume="approve"), config=config)
                snapshot = pipeline.get_state(config)

        if snapshot.tasks and snapshot.tasks[0].interrupts:
            outlier_interrupt = snapshot.tasks[0].interrupts[0].value
            results["outlier_proposals"] = outlier_interrupt.get("proposals", {})
            state = pipeline.invoke(Command(resume="approve"), config=config)
        else:
            state = snapshot.values

        results["report"] = state.get("report", {})
        results["quality_score"] = state.get("quality_score", {})
        results["confidence_map"] = state.get("confidence_map", [])
        results["final_state"] = state
        results["success"] = True

    except Exception as e:
        results["success"] = False
        results["errors"].append(f"{type(e).__name__}: {e}")

    results["duration"] = round(time.time() - start, 1)
    return results

## 2. Titanic — Walkthrough complet

Dataset classique (891 lignes, 15 colonnes). Teste :
- Valeurs manquantes (age, deck, embarked)
- Mix numerique / categorique
- Classification binaire (survived)
- Detection automatique de colonnes triviales

In [3]:
url_titanic = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df_titanic = pd.read_csv(url_titanic)

print(f"Shape : {df_titanic.shape}")
print(f"\nTypes :\n{df_titanic.dtypes.value_counts()}")
print(f"\nValeurs manquantes :")
missing = df_titanic.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))
df_titanic.head()

Shape : (891, 15)

Types :
str        7
int64      4
float64    2
bool       2
Name: count, dtype: int64

Valeurs manquantes :
deck           688
age            177
embarked         2
embark_town      2
dtype: int64


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### 2.1 Execution du pipeline

In [4]:
results_titanic = run_pipeline_auto(url_titanic, "survived", thread_id="titanic")
print(f"Succes : {results_titanic['success']} | Duree : {results_titanic['duration']}s")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: latifo (latifo-universit-paris-dauphine-psl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
18:18:31 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=titanic
18:18:31 - src.agents.base_agent - INFO - Dataset loaded: 891 rows x 15 columns, target='survived'
18:18:31 - src.agents.base_agent - INFO - Target 'survived': task_type=classification, 2 unique, 0.0% NaN
18:18:31 - src.agents.base_agent - INFO - Dtype audit: 3 columns mistyped
18:18:31 - src.agents.base_agent - WARNING - Missing pattern: 'embarked' and 'embark_town' have correlated missingness (r=1.00) -- potential MAR/MNAR, mean/median imputation may introduce bias
18:18:31 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:18:38 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
1

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.6145
baseline/delta,0.3855
baseline/model_score,1


18:19:59 - src.tracking.wandb_tracker - INFO - W&B run finished


Succes : True | Duree : 97.8s


### 2.2 Analyse du domaine infere

In [5]:
domain = results_titanic.get("domain_context", {})
print(f"Domaine : {domain.get('domain', '?')}")
print(f"Description : {domain.get('description', '')}")
print(f"\nContraintes du domaine :")
for c in domain.get("constraints", []):
    print(f"  - {c}")
print(f"\nColonnes sensibles : {domain.get('sensitive_columns', [])}")

anomalies = results_titanic.get("semantic_anomalies", [])
if anomalies:
    print(f"\nAnomalies semantiques detectees : {len(anomalies)}")
    for a in anomalies:
        print(f"  {a['column']} [{a['severity']}] : {a['issue']}")
        if a.get("sentinel_values"):
            print(f"    Valeurs sentinelles : {a['sentinel_values']}")

Domaine : transport
Description : Ce dataset contient des informations sur les passagers du Titanic, y compris leur statut de survie, classe, sexe, âge, et d'autres caractéristiques. Il est utilisé pour analyser les facteurs influençant la survie lors du naufrage.

Contraintes du domaine :
  - l'âge ne peut pas être négatif
  - le tarif doit être positif
  - le nombre de frères/sœurs ou conjoints à bord (sibsp) ne peut pas être négatif
  - le nombre de parents ou enfants à bord (parch) ne peut pas être négatif

Colonnes sensibles : ['sex', 'age', 'deck', 'embark_town']

Anomalies semantiques detectees : 3
  age [high] : Âges négatifs détectés
  fare [medium] : Tarifs à 0 détectés
  deck [medium] : Valeurs manquantes excessives (77.2% de NaN)


### 2.3 Transformations proposees

In [6]:
transforms = results_titanic.get("transform_proposals", {}).get("transformations", [])

df_transforms = pd.DataFrame([
    {
        "Colonne": t["column"],
        "Action": t["action"],
        "Source": t.get("source", "llm"),
        "Confiance": t.get("confidence", "?"),
        "Raison": t.get("reason", "")[:80],
    }
    for t in transforms
])

rule_count = (df_transforms["Source"] == "rule").sum() if len(df_transforms) else 0
llm_count = len(df_transforms) - rule_count
print(f"Total : {len(df_transforms)} transformations ({rule_count} rule, {llm_count} LLM)")
df_transforms

Total : 19 transformations (7 rule, 12 LLM)


,Colonne,Action,Source,Confiance,Raison
0,deck,drop_column,rule,high,78% NaN (>50%)
1,age,impute_median,llm,medium,"19.9% des valeurs sont manquantes, imputation ..."
2,deck,impute_mode,llm,medium,"77.2% des valeurs sont manquantes, imputation ..."
3,pclass,ordinal_encoding,rule,medium,"numeric with 3 unique values, range=2 (<= 6)"
4,sibsp,ordinal_encoding,rule,medium,"numeric with 7 unique values, range=8 (<= 14)"
5,parch,ordinal_encoding,rule,medium,"numeric with 7 unique values, range=6 (<= 14)"
6,adult_male,ordinal_encoding,rule,high,boolean column -> int
7,alone,ordinal_encoding,rule,high,boolean column -> int
8,pclass,ordinal_encoding,llm,medium,Transformation en type ordinal pour représente...
9,parch,ordinal_encoding,llm,medium,Transformation en type ordinal pour représente...


### 2.4 Leakage et correlations

In [7]:
leakage = results_titanic.get("leakage_alerts", [])
correlations = results_titanic.get("correlation_alerts", [])

if leakage:
    print(f"Alertes leakage : {len(leakage)}")
    for l in leakage:
        print(f"  {l['column']} : corr={l['correlation']:.3f} — {l['reason']}")
else:
    print("Aucune alerte de leakage")

if correlations:
    print(f"\nMulticollinearite : {len(correlations)} paire(s)")
    for c in correlations:
        method_label = "Pearson" if c["method"] == "pearson" else "Cramer's V"
        print(f"  {c['column_a']} <-> {c['column_b']} : {c['correlation']:.3f} ({method_label}, {c['severity']})")
        for s in c.get("suggested_actions", []):
            print(f"    -> {s}")
else:
    print("\nAucune alerte de multicollinearite")

Aucune alerte de leakage

Multicollinearite : 4 paire(s)
  sex <-> who : 0.948 (Cramer's V, very_high)
    -> drop_least_informative: supprimer la variable avec le moins de variance ou la moins corrélée au target
    -> pca: réduction de dimension sur le groupe de variables corrélées (>= 3)
    -> keep: certains modèles (arbres, gradient boosting) tolèrent la colinéarité
  sex <-> adult_male : 0.908 (Cramer's V, very_high)
    -> drop_least_informative: supprimer la variable avec le moins de variance ou la moins corrélée au target
    -> pca: réduction de dimension sur le groupe de variables corrélées (>= 3)
    -> keep: certains modèles (arbres, gradient boosting) tolèrent la colinéarité
  embarked <-> embark_town : 1.000 (Cramer's V, very_high)
    -> drop_least_informative: supprimer la variable avec le moins de variance ou la moins corrélée au target
    -> pca: réduction de dimension sur le groupe de variables corrélées (>= 3)
    -> keep: certains modèles (arbres, gradient boosti

### 2.5 Outliers

In [8]:
outlier_actions = results_titanic.get("outlier_proposals", {}).get("outlier_actions", [])

if outlier_actions:
    df_outliers = pd.DataFrame([
        {
            "Colonne": o["column"],
            "Methode": o["method"],
            "Action": o["action"],
            "Source": o.get("source", "llm"),
            "Confiance": o.get("confidence", "?"),
            "Review": o.get("requires_review", False),
        }
        for o in outlier_actions
    ])
    print(f"Outlier actions : {len(df_outliers)}")
    display(df_outliers)
else:
    print("Aucune action outlier proposee")

Outlier actions : 9


,Colonne,Methode,Action,Source,Confiance,Review
0,pclass,bimodal,keep,rule,high,False
1,sibsp,bimodal,keep,rule,high,False
2,parch,bimodal,keep,rule,high,False
3,fare,bimodal,keep,rule,high,False
4,adult_male,bimodal,keep,rule,high,False
5,alive,bimodal,keep,rule,high,False
6,alone,bimodal,keep,rule,high,False
7,age_was_imputed,bimodal,keep,rule,high,False
8,age,iqr,replace_with_nan,rule,high,False


### 2.6 Quality score et rapport

In [9]:
qs = results_titanic.get("quality_score", {})
report = results_titanic.get("report", {})
ba = report.get("before_after", {})

print(f"Quality score : {qs.get('overall', '?')}/100")
print(f"  Completeness     : {qs.get('completeness', '?')}/30")
print(f"  Type consistency : {qs.get('type_consistency', '?')}/15")
print(f"  Leakage risk     : {qs.get('leakage_risk', '?')}/15")
print(f"  Outlier coverage : {qs.get('outlier_coverage', '?')}/20")
print(f"  Drop penalty     : {qs.get('drop_penalty', 0)}")

print(f"\nShape : {ba.get('original_shape', '?')} -> {ba.get('final_shape', '?')}")
print(f"Reduction NaN : {ba.get('missing_reduction', '?')}")

# Confidence map
conf_map = results_titanic.get("confidence_map", [])
if conf_map:
    total = len(conf_map)
    rule_ct = sum(1 for c in conf_map if c.get("source") == "rule")
    low_conf = [c["column"] for c in conf_map if c.get("confidence") == "low"]
    print(f"\nRatio deterministe : {rule_ct}/{total} ({rule_ct/total*100:.0f}%)")
    if low_conf:
        print(f"Colonnes basse confiance : {low_conf}")

Quality score : 85.8/100
  Completeness     : 29.8/30
  Type consistency : 6.0/15
  Leakage risk     : low/15
  Outlier coverage : 20.0/20
  Drop penalty     : 5.0

Shape : 891 rows x 15 columns -> 712 rows x 20 columns
Reduction NaN : ?

Ratio deterministe : 16/27 (59%)


### 2.7 Baseline evaluation

In [10]:
baseline = report.get("baseline_evaluation", {})
if baseline:
    print(f"Metrique : {baseline.get('metric', '?')}")
    print(f"Baseline (dummy)  : {baseline.get('baseline_score', '?'):.4f}")
    print(f"Modele simple     : {baseline.get('model_score', '?'):.4f}")
    print(f"Delta             : +{baseline.get('delta', 0):.4f}")
else:
    print("Pas d'evaluation baseline disponible")

Metrique : accuracy
Baseline (dummy)  : 0.6145
Modele simple     : 1.0000
Delta             : +0.3855


### 2.8 Donnees transformees

In [11]:
final_state = results_titanic.get("final_state", {})
train_path = final_state.get("train_path", "")
test_path = final_state.get("test_path", "")

if train_path and Path(train_path).exists():
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    print(f"Train : {df_train.shape}, Test : {df_test.shape}")
    print(f"NaN restants (train) : {df_train.isnull().sum().sum()}")
    print(f"NaN restants (test)  : {df_test.isnull().sum().sum()}")
    print(f"\nColonnes finales : {list(df_train.columns)}")
    df_train.head()

Train : (712, 20), Test : (179, 20)
NaN restants (train) : 29
NaN restants (test)  : 11

Colonnes finales : ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'adult_male', 'alive', 'alone', 'age_was_imputed', 'sex_male', 'embarked_Q', 'embarked_S', 'class_Second', 'class_Third', 'who_man', 'who_woman', 'embark_town_Queenstown', 'embark_town_Southampton', 'age_was_outlier']


### 2.9 Artefacts sauvegardes

In [12]:
artifacts_dir = Path("data/outputs/artifacts")
if artifacts_dir.exists():
    artifacts = sorted(artifacts_dir.glob("*"))
    print(f"Artefacts sauvegardes ({len(artifacts)}) :")
    for a in artifacts:
        size_kb = a.stat().st_size / 1024
        print(f"  {a.name} ({size_kb:.1f} KB)")
else:
    print("Aucun artefact trouve")

Artefacts sauvegardes (68) :
  freq_map_high_cardinality_id.joblib (7.0 KB)
  impute_mean_bill_depth_mm.joblib (0.0 KB)
  impute_mean_bill_length_mm.joblib (0.0 KB)
  impute_mean_body_mass_g.joblib (0.0 KB)
  impute_mean_flipper_length_mm.joblib (0.0 KB)
  impute_median_age.joblib (0.0 KB)
  impute_median_bill_depth_mm.joblib (0.0 KB)
  impute_median_bill_length_mm.joblib (0.0 KB)
  impute_median_body_mass_g.joblib (0.0 KB)
  impute_median_flipper_length_mm.joblib (0.0 KB)
  impute_median_normal_col.joblib (0.0 KB)
  impute_mode_sex.joblib (0.0 KB)
  label_encoder_adult_male.joblib (0.5 KB)
  label_encoder_alive.joblib (0.5 KB)
  label_encoder_binary_cat.joblib (0.5 KB)
  label_encoder_sex.joblib (0.5 KB)
  log_shift_fare.joblib (0.0 KB)
  ohe_categories_alive.joblib (0.0 KB)
  ohe_categories_b.joblib (0.0 KB)
  ohe_categories_binary_cat.joblib (0.0 KB)
  ohe_categories_clarity.joblib (0.1 KB)
  ohe_categories_class.joblib (0.0 KB)
  ohe_categories_color.joblib (0.1 KB)
  ohe_categorie

## 3. Tips — Dataset propre, regression

244 lignes, 7 colonnes, aucune valeur manquante. Regression sur `tip`.
Teste si l'agent evite de sur-transformer un dataset deja propre.

In [13]:
url_tips = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df_tips = pd.read_csv(url_tips)

print(f"Shape : {df_tips.shape}")
print(f"Types : {dict(df_tips.dtypes.value_counts())}")
print(f"NaN total : {df_tips.isnull().sum().sum()}")
df_tips.head(3)

Shape : (244, 7)
Types : {<StringDtype(storage='python', na_value=nan)>: np.int64(4), dtype('float64'): np.int64(2), dtype('int64'): np.int64(1)}
NaN total : 0


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3


In [14]:
results_tips = run_pipeline_auto(url_tips, "tip", thread_id="tips")
print(f"Succes : {results_tips['success']} | Duree : {results_tips['duration']}s")

18:20:01 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=tips
18:20:01 - src.agents.base_agent - INFO - Dataset loaded: 244 rows x 7 columns, target='tip'
18:20:01 - src.agents.base_agent - INFO - Target 'tip': task_type=regression, 123 unique, 0.0% NaN
18:20:01 - src.agents.base_agent - INFO - Dtype audit: 1 columns mistyped
18:20:01 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:20:04 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
18:20:12 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'summary': Input should be a valid string (valeur recue: {'shape': '244 rows x 7 columns', 'target': 'tip (regression)', 'domain': 'e-commerce', 'columns': {'total_bill': {'type': 'float64', 'min': 3.07, 'max': 50.81, 'nunique': 229, 'skewness': 1.133, 'outlier_ratio': 3.7}, 'sex': {'type': 'str'}, 'smoker': {'type': 'str'}, 'day': {'type': 'str'}, 'time': {'type': 

Succes : False | Duree : 29.2s


In [15]:
# Resume compact
transforms_tips = results_tips.get("transform_proposals", {}).get("transformations", [])
outliers_tips = results_tips.get("outlier_proposals", {}).get("outlier_actions", [])
qs_tips = results_tips.get("quality_score", {})
ba_tips = results_tips.get("report", {}).get("before_after", {})

print(f"Domaine : {results_tips.get('domain_context', {}).get('domain', '?')}")
print(f"Transformations : {len(transforms_tips)}")
for t in transforms_tips:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")
print(f"Outlier actions : {len(outliers_tips)}")
for o in outliers_tips:
    print(f"  [{o.get('source', 'llm')}] {o['column']} : {o['method']}/{o['action']}")
print(f"Quality score : {qs_tips.get('overall', '?')}/100")
print(f"Shape : {ba_tips.get('original_shape', '?')} -> {ba_tips.get('final_shape', '?')}")

Domaine : ?
Transformations : 0
Outlier actions : 0
Quality score : ?/100
Shape : ? -> ?


## 4. Diamonds — Grand dataset, haute cardinalite

53 940 lignes, colonnes ordinales (`cut`, `color`, `clarity`), regression sur `price`.
Teste :
- Performance sur un gros volume
- Gestion de la haute cardinalite (frequency encoding vs one-hot)
- Detection d'ordinaux

In [16]:
url_diamonds = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
df_diamonds = pd.read_csv(url_diamonds)

print(f"Shape : {df_diamonds.shape}")
print(f"\nCardinalite des categoriques :")
for col in df_diamonds.select_dtypes(include="object").columns:
    print(f"  {col} : {df_diamonds[col].nunique()} valeurs uniques — {list(df_diamonds[col].unique()[:8])}")
df_diamonds.head(3)

Shape : (53940, 10)

Cardinalite des categoriques :
  cut : 5 valeurs uniques — ['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
  color : 7 valeurs uniques — ['E', 'I', 'J', 'H', 'F', 'G', 'D']
  clarity : 8 valeurs uniques — ['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']


C:\Users\abdel\AppData\Local\Temp\ipykernel_27384\3679635.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_diamonds.select_dtypes(include="object").columns:


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31


In [17]:
results_diamonds = run_pipeline_auto(url_diamonds, "price", thread_id="diamonds")
print(f"Succes : {results_diamonds['success']} | Duree : {results_diamonds['duration']}s")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


18:20:34 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=diamonds
18:20:35 - src.agents.base_agent - INFO - Dataset loaded: 53940 rows x 10 columns, target='price'
18:20:35 - src.agents.base_agent - INFO - Target 'price': task_type=regression, 11602 unique, 0.0% NaN
18:20:35 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:20:41 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
18:20:51 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'summary': Input should be a valid string (valeur recue: {'shape': '53940 rows x 10 columns', 'target': 'price', 'domain': 'e-commerce - informations sur des diamants', 'columns': {'carat': {'type': 'float64', 'min': 0.2, 'max': 5.01, 'nunique': 273, 'skewness': 1.117, 'outlier_ratio': 3.5}, 'cut': {'type': 'str'}, 'color': {'type': 'str'}, 'clarity': {'type': 'str'}, 'depth': {'type': 'float64', 'min': 43.0, 'max': 79.0, 'nu

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0
baseline/delta,0.8615
baseline/model_score,0.8615


18:22:12 - src.tracking.wandb_tracker - INFO - W&B run finished


Succes : True | Duree : 102.0s


In [18]:
transforms_dia = results_diamonds.get("transform_proposals", {}).get("transformations", [])
outliers_dia = results_diamonds.get("outlier_proposals", {}).get("outlier_actions", [])
qs_dia = results_diamonds.get("quality_score", {})
ba_dia = results_diamonds.get("report", {}).get("before_after", {})

print(f"Domaine : {results_diamonds.get('domain_context', {}).get('domain', '?')}")
print(f"\nTransformations : {len(transforms_dia)}")
for t in transforms_dia:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")
print(f"\nOutlier actions : {len(outliers_dia)}")
for o in outliers_dia:
    print(f"  [{o.get('source', 'llm')}] {o['column']} : {o['method']}/{o['action']}")
print(f"\nQuality score : {qs_dia.get('overall', '?')}/100")
print(f"Shape : {ba_dia.get('original_shape', '?')} -> {ba_dia.get('final_shape', '?')}")

# Verifier que les ordinaux sont bien detectes
ordinal_cols = [t for t in transforms_dia if t["action"] == "ordinal_encoding"]
freq_cols = [t for t in transforms_dia if t["action"] == "frequency_encoding"]
print(f"\nOrdinal encoding : {[t['column'] for t in ordinal_cols]}")
print(f"Frequency encoding : {[t['column'] for t in freq_cols]}")

Domaine : e-commerce

Transformations : 21
  [llm] carat : drop_rows
  [llm] depth : drop_rows
  [llm] table : drop_rows
  [llm] x : drop_rows
  [llm] y : drop_rows
  [llm] z : drop_rows
  [llm] cut : one_hot_encoding
  [llm] color : one_hot_encoding
  [llm] clarity : one_hot_encoding
  [rule] carat : robust_scaling
  [rule] depth : standard_scaling
  [rule] table : standard_scaling
  [rule] x : standard_scaling
  [rule] y : log_transform
  [rule] z : log_transform
  [llm] depth : robust_scaling
  [llm] table : robust_scaling
  [llm] carat : robust_scaling
  [llm] x : robust_scaling
  [llm] y : robust_scaling
  [llm] z : robust_scaling

Outlier actions : 6
  [rule] x : iqr/replace_with_nan
  [rule] y : iqr/replace_with_nan
  [rule] z : iqr/replace_with_nan
  [llm] carat : iqr/keep
  [llm] depth : iqr/remove
  [llm] table : iqr/keep

Quality score : 98.3/100
Shape : (53940, 10) -> (41132, 27)

Ordinal encoding : []
Frequency encoding : []


## 5. Penguins — Multi-classe avec NaN

344 lignes, 3 especes de pingouins, NaN disperses dans colonnes numeriques et categoriques.
Teste :
- Classification multi-classe (3 classes)
- Stratified split avec petits groupes
- Imputation sur donnees avec peu de lignes

In [19]:
url_penguins = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df_penguins = pd.read_csv(url_penguins)

print(f"Shape : {df_penguins.shape}")
print(f"Target distribution : {dict(df_penguins['species'].value_counts())}")
print(f"\nNaN :")
missing_p = df_penguins.isnull().sum()
print(missing_p[missing_p > 0])
df_penguins.head(3)

Shape : (344, 7)
Target distribution : {'Adelie': np.int64(152), 'Gentoo': np.int64(124), 'Chinstrap': np.int64(68)}

NaN :
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE


In [20]:
results_penguins = run_pipeline_auto(url_penguins, "species", thread_id="penguins")
print(f"Succes : {results_penguins['success']} | Duree : {results_penguins['duration']}s")

18:22:16 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=penguins
18:22:16 - src.agents.base_agent - INFO - Dataset loaded: 344 rows x 7 columns, target='species'
18:22:16 - src.agents.base_agent - INFO - Target 'species': task_type=classification, 3 unique, 0.0% NaN
18:22:16 - src.agents.base_agent - WARNING - Missing pattern: 'bill_length_mm' and 'bill_depth_mm' have correlated missingness (r=1.00) -- potential MAR/MNAR, mean/median imputation may introduce bias
18:22:16 - src.agents.base_agent - WARNING - Missing pattern: 'bill_length_mm' and 'flipper_length_mm' have correlated missingness (r=1.00) -- potential MAR/MNAR, mean/median imputation may introduce bias
18:22:16 - src.agents.base_agent - WARNING - Missing pattern: 'bill_length_mm' and 'body_mass_g' have correlated missingness (r=1.00) -- potential MAR/MNAR, mean/median imputation may introduce bias
18:22:16 - src.agents.base_agent - WARNING - Missing pattern: 'bill_depth_mm' and 'f

Succes : False | Duree : 28.9s


In [21]:
transforms_pen = results_penguins.get("transform_proposals", {}).get("transformations", [])
qs_pen = results_penguins.get("quality_score", {})
meta_pen = results_penguins.get("final_state", {}).get("target_meta", {})

print(f"Domaine : {results_penguins.get('domain_context', {}).get('domain', '?')}")
print(f"Task type : {meta_pen.get('task_type', '?')}, n_classes : {meta_pen.get('n_classes', '?')}")
print(f"\nTransformations : {len(transforms_pen)}")
for t in transforms_pen:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")
print(f"\nQuality score : {qs_pen.get('overall', '?')}/100")

Domaine : ?
Task type : ?, n_classes : ?

Transformations : 0

Quality score : ?/100


## 6. Cas extremes — Dataset synthetique

Dataset fabrique pour pousser l'agent dans ses limites :

| Colonne | Defi |
|---------|------|
| `all_nan_col` | 100% NaN — doit etre auto-drop |
| `constant_col` | Variance 0 — doit etre auto-drop |
| `high_cardinality_id` | 500 valeurs uniques (IDs) — doit etre auto-drop |
| `extreme_values` | Outliers extremes (1e8) — detection robuste |
| `mostly_nan_col` | 95% NaN — doit etre drop (>50% threshold) |
| `datetime_col` | Type datetime — gestion des types non standards |
| `binary_cat` | Categorique binaire simple |

In [22]:
np.random.seed(42)
n = 500

df_extreme = pd.DataFrame({
    "target": np.random.choice([0, 1], size=n),
    "normal_col": np.random.randn(n),
    "all_nan_col": [np.nan] * n,
    "constant_col": [42] * n,
    "high_cardinality_id": [f"ID_{i}" for i in range(n)],
    "extreme_values": np.concatenate([np.random.randn(n - 5), [1e6, -1e6, 1e7, -1e7, 1e8]]),
    "binary_cat": np.random.choice(["yes", "no"], size=n),
    "datetime_col": pd.date_range("2020-01-01", periods=n, freq="h"),
    "mostly_nan_col": np.where(np.random.rand(n) < 0.95, np.nan, np.random.randn(n)),
})

extreme_path = "data/outputs/extreme_test.csv"
Path(extreme_path).parent.mkdir(parents=True, exist_ok=True)
df_extreme.to_csv(extreme_path, index=False)

print(f"Shape : {df_extreme.shape}")
print(f"\nValeurs manquantes :")
print(df_extreme.isnull().sum())
print(f"\nCardinalite :")
for col in df_extreme.select_dtypes(include="object").columns:
    print(f"  {col} : {df_extreme[col].nunique()}")
df_extreme.head(3)

Shape : (500, 9)

Valeurs manquantes :
target                   0
normal_col               0
all_nan_col            500
constant_col             0
high_cardinality_id      0
extreme_values           0
binary_cat               0
datetime_col             0
mostly_nan_col         477
dtype: int64

Cardinalite :
  high_cardinality_id : 500
  binary_cat : 2


C:\Users\abdel\AppData\Local\Temp\ipykernel_27384\2795033435.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_extreme.select_dtypes(include="object").columns:


,target,normal_col,all_nan_col,constant_col,high_cardinality_id,extreme_values,binary_cat,datetime_col,mostly_nan_col
0,0,-0.846794,NaN,42,ID_0,0.710960,no,2020-01-01 00:00:00,NaN
1,1,-1.514847,NaN,42,ID_1,0.444263,no,2020-01-01 01:00:00,NaN
2,0,-0.446515,NaN,42,ID_2,-0.360966,yes,2020-01-01 02:00:00,NaN


In [23]:
results_extreme = run_pipeline_auto(extreme_path, "target", thread_id="extreme")
print(f"Succes : {results_extreme['success']} | Duree : {results_extreme['duration']}s")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


18:22:44 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=extreme_test
18:22:44 - src.agents.base_agent - INFO - Dataset loaded: 500 rows x 9 columns, target='target'
18:22:44 - src.agents.base_agent - INFO - Target 'target': task_type=classification, 2 unique, 0.0% NaN
18:22:44 - src.agents.base_agent - INFO - Dtype audit: 2 columns mistyped
18:22:44 - src.agents.base_agent - INFO - Auto-drop: ['all_nan_col', 'constant_col']
18:22:44 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:22:49 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
18:23:01 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'summary': Input should be a valid string (valeur recue: {'shape': '500 rows x 9 columns', 'target': 'binary classification', 'columns': {'normal_col': {'type': 'float64', 'min': -3.2413, 'max': 3.8527, 'nunique': 500, 'skewness': 0.138, 'outlier_ratio': 0.6}, 'high_

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.51
baseline/delta,0.02
baseline/model_score,0.53


18:23:57 - src.tracking.wandb_tracker - INFO - W&B run finished


Succes : True | Duree : 74.8s


In [24]:
# Verification des colonnes triviales auto-drop
transforms_ext = results_extreme.get("transform_proposals", {}).get("transformations", [])

expected_drops = {
    "all_nan_col": "100% NaN",
    "constant_col": "variance 0",
    "high_cardinality_id": "IDs uniques",
    "mostly_nan_col": ">50% NaN",
}

print("Verification auto-drop :")
for col_name, reason in expected_drops.items():
    match = [t for t in transforms_ext if t["column"] == col_name and t["action"] == "drop_column"]
    status = "OK" if match else "MANQUE"
    source = match[0].get("source", "?") if match else "-"
    print(f"  [{status}] {col_name} ({reason}) — source: {source}")

# Datetime
dt_actions = [t for t in transforms_ext if t["column"] == "datetime_col"]
if dt_actions:
    print(f"\nDatetime : {dt_actions[0]['action']} (source: {dt_actions[0].get('source', '?')})")
else:
    print(f"\nDatetime : non traitee (ignoree)")

qs_ext = results_extreme.get("quality_score", {})
print(f"\nQuality score : {qs_ext.get('overall', '?')}/100")

Verification auto-drop :
  [OK] all_nan_col (100% NaN) — source: rule
  [OK] constant_col (variance 0) — source: rule
  [MANQUE] high_cardinality_id (IDs uniques) — source: -
  [OK] mostly_nan_col (>50% NaN) — source: rule

Datetime : extract_datetime (source: llm)

Quality score : 88.9/100


## 7. Adult Census — Valeurs sentinelles, grand dataset

32 561 lignes, 14 features. Classification binaire (income >50K / <=50K).

Points d'interet :
- Valeurs sentinelles " ?" dans `workclass`, `occupation`, `native-country`
- Haute cardinalite sur `native-country` (41 valeurs) et `occupation` (14)
- Colonnes numeriques + categoriques melangees
- Dataset desequilibre (~75% <=50K)

In [25]:
url_adult = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
col_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income",
]
df_adult = pd.read_csv(url_adult, names=col_names, skipinitialspace=True)

print(f"Shape : {df_adult.shape}")
print(f"\nValeurs '?' (sentinelles) :")
for col in df_adult.columns:
    q_count = (df_adult[col] == "?").sum()
    if q_count > 0:
        print(f"  {col} : {q_count} ({q_count/len(df_adult)*100:.1f}%)")
print(f"\nDistribution target :\n{df_adult['income'].value_counts()}")
print(f"\nCardinalite categoriques :")
for col in df_adult.select_dtypes(include="object").columns:
    if col != "income":
        print(f"  {col} : {df_adult[col].nunique()}")
df_adult.head(3)

Shape : (48842, 15)

Valeurs '?' (sentinelles) :
  workclass : 2799 (5.7%)
  occupation : 2809 (5.8%)
  native-country : 857 (1.8%)

Distribution target :
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Cardinalite categoriques :
  workclass : 9
  education : 16
  marital-status : 7
  occupation : 15
  relationship : 6
  race : 5
  sex : 2
  native-country : 42


C:\Users\abdel\AppData\Local\Temp\ipykernel_27384\2825181030.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_adult.select_dtypes(include="object").columns:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


In [26]:
# Sauvegarder localement (le pipeline attend un path ou URL)
adult_path = "data/outputs/adult_census.csv"
df_adult.to_csv(adult_path, index=False)

results_adult = run_pipeline_auto(adult_path, "income", thread_id="adult")
print(f"Succes : {results_adult['success']} | Duree : {results_adult['duration']}s")

18:24:01 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=adult_census
18:24:02 - src.agents.base_agent - INFO - Dataset loaded: 48842 rows x 15 columns, target='income'
18:24:02 - src.agents.base_agent - INFO - Target 'income': task_type=classification, 2 unique, 0.0% NaN
18:24:02 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:24:08 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
18:24:26 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'summary': Input should be a valid string (valeur recue: {'shape': '48842 rows x 15 columns', 'target': 'income (classification)', 'domain': 'Données démographiques et économiques pour prédire le revenu annuel.', 'columns': {'age': {'type': 'int64', 'min': 17, 'max': 90, 'nunique': 74, 'skewness': 0.558, 'outlier_ratio': 0.4}, 'workclass': {'type': 'str'}, 'fnlwgt': {'type': 'int64', 'min': 12285, 'max': 1490400, 'nuniq

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.7607
baseline/delta,0.0573
baseline/model_score,0.818


18:26:08 - src.tracking.wandb_tracker - INFO - W&B run finished


Succes : True | Duree : 128.4s


In [27]:
transforms_adult = results_adult.get("transform_proposals", {}).get("transformations", [])
outliers_adult = results_adult.get("outlier_proposals", {}).get("outlier_actions", [])
qs_adult = results_adult.get("quality_score", {})
ba_adult = results_adult.get("report", {}).get("before_after", {})

print(f"Domaine : {results_adult.get('domain_context', {}).get('domain', '?')}")
print(f"\nTransformations : {len(transforms_adult)}")
for t in transforms_adult:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")

# Verifier si les "?" ont ete detectes comme sentinelles
sentinel_transforms = [t for t in transforms_adult if t["action"] == "replace_sentinel"]
print(f"\nReplace_sentinel : {len(sentinel_transforms)}")
for t in sentinel_transforms:
    print(f"  {t['column']} — {t.get('reason', '')[:80]}")

print(f"\nOutlier actions : {len(outliers_adult)}")
print(f"Quality score : {qs_adult.get('overall', '?')}/100")
print(f"Shape : {ba_adult.get('original_shape', '?')} -> {ba_adult.get('final_shape', '?')}")

Domaine : autre

Transformations : 18
  [llm] age : drop_rows
  [llm] fnlwgt : drop_rows
  [llm] capital-gain : drop_rows
  [llm] hours-per-week : drop_rows
  [llm] workclass : one_hot_encoding
  [llm] education : one_hot_encoding
  [llm] marital-status : one_hot_encoding
  [llm] occupation : one_hot_encoding
  [llm] relationship : one_hot_encoding
  [llm] race : one_hot_encoding
  [llm] sex : label_encoding
  [llm] native-country : one_hot_encoding
  [rule] age : standard_scaling
  [rule] fnlwgt : robust_scaling
  [rule] education-num : standard_scaling
  [rule] capital-gain : log_transform
  [rule] capital-loss : log_transform
  [rule] hours-per-week : standard_scaling

Replace_sentinel : 0

Outlier actions : 7
Quality score : 98.5/100
Shape : ? -> ?


## 8. Mpg — Numerique-as-string, petit dataset

398 lignes, 9 colonnes. Regression sur `mpg` (miles per gallon).

Points d'interet :
- `horsepower` stocke comme string avec des "?" — teste `cast_numeric` + detection sentinelle
- Petit dataset (< 400 lignes)
- Domaine automobile
- Colonne `name` = identifiant texte (doit etre auto-drop)

In [28]:
url_mpg = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
df_mpg = pd.read_csv(url_mpg)

print(f"Shape : {df_mpg.shape}")
print(f"\nTypes :\n{df_mpg.dtypes}")
print(f"\nNaN :")
missing_mpg = df_mpg.isnull().sum()
print(missing_mpg[missing_mpg > 0])
print(f"\nColonne 'name' (ex) : {df_mpg['name'].head(3).tolist()}")
print(f"Cardinalite 'name' : {df_mpg['name'].nunique()} / {len(df_mpg)}")
df_mpg.head(3)

Shape : (398, 9)

Types :
mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin              str
name                str
dtype: object

NaN :
horsepower    6
dtype: int64

Colonne 'name' (ex) : ['chevrolet chevelle malibu', 'buick skylark 320', 'plymouth satellite']
Cardinalite 'name' : 305 / 398


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite


In [29]:
results_mpg = run_pipeline_auto(url_mpg, "mpg", thread_id="mpg")
print(f"Succes : {results_mpg['success']} | Duree : {results_mpg['duration']}s")

18:26:11 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=mpg
18:26:11 - src.agents.base_agent - INFO - Dataset loaded: 398 rows x 9 columns, target='mpg'
18:26:11 - src.agents.base_agent - INFO - Target 'mpg': task_type=regression, 129 unique, 0.0% NaN
18:26:11 - src.agents.base_agent - INFO - Dtype audit: 1 columns mistyped
18:26:11 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
18:26:15 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
18:26:26 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'summary': Input should be a valid string (valeur recue: {'shape': '398 rows x 9 columns', 'target': 'mpg', 'domain': 'transport', 'description': 'Le dataset contient des informations sur les véhicules, y compris leur consommation de carburant (mpg), caractéristiques techniques et origine. Il est utilisé pour prédire la consommation de carburant en fonction de divers

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,-0.004
baseline/delta,0.8432
baseline/model_score,0.8392


18:27:15 - src.tracking.wandb_tracker - INFO - W&B run finished


Succes : True | Duree : 66.1s


In [30]:
transforms_mpg = results_mpg.get("transform_proposals", {}).get("transformations", [])
qs_mpg = results_mpg.get("quality_score", {})
ba_mpg = results_mpg.get("report", {}).get("before_after", {})

print(f"Domaine : {results_mpg.get('domain_context', {}).get('domain', '?')}")
print(f"\nTransformations : {len(transforms_mpg)}")
for t in transforms_mpg:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")

# Verifier que 'name' est auto-drop (haute cardinalite / ID)
name_drop = any(t["column"] == "name" and t["action"] == "drop_column" for t in transforms_mpg)
print(f"\nAuto-drop 'name' : {'OK' if name_drop else 'MANQUE'}")

# Verifier horsepower
hp_actions = [t for t in transforms_mpg if t["column"] == "horsepower"]
if hp_actions:
    print(f"Horsepower : {[t['action'] for t in hp_actions]}")

print(f"\nQuality score : {qs_mpg.get('overall', '?')}/100")
print(f"Shape : {ba_mpg.get('original_shape', '?')} -> {ba_mpg.get('final_shape', '?')}")

Domaine : transport

Transformations : 13
  [rule] horsepower : impute_median
  [llm] horsepower : impute_mean
  [llm] name : drop_column
  [rule] cylinders : ordinal_encoding
  [llm] cylinders : ordinal_encoding
  [llm] origin : one_hot_encoding
  [rule] displacement : standard_scaling
  [rule] horsepower : robust_scaling
  [rule] weight : standard_scaling
  [rule] acceleration : standard_scaling
  [rule] model_year : standard_scaling
  [llm] horsepower : log_transform
  [llm] displacement : log_transform

Auto-drop 'name' : OK
Horsepower : ['impute_median', 'impute_mean', 'robust_scaling', 'log_transform']

Quality score : 92.0/100
Shape : (398, 9) -> (318, 11)


## 9. Wine Quality — Tout numerique, pas de NaN

4 898 lignes (red + white), 12 colonnes toutes numeriques sauf `color`.
Regression sur `quality` (score 3-9).

Points d'interet :
- Pas de valeurs manquantes — teste si l'agent evite les transformations inutiles
- Beaucoup de colonnes numeriques — teste les choix de scaling (standard vs robust vs log)
- Distribution de la target concentree (5-7) — detection d'outliers pertinente
- `color` binaire (red/white) — encodage simple attendu

In [31]:
url_wine_red = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/winequality-red.csv"
url_wine_white = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/winequality-white.csv"

df_red = pd.read_csv(url_wine_red, sep=";")
df_red["color"] = "red"
df_white = pd.read_csv(url_wine_white, sep=";")
df_white["color"] = "white"
df_wine = pd.concat([df_red, df_white], ignore_index=True)

wine_path = "data/outputs/wine_quality.csv"
df_wine.to_csv(wine_path, index=False)

print(f"Shape : {df_wine.shape}")
print(f"Types : {dict(df_wine.dtypes.value_counts())}")
print(f"NaN total : {df_wine.isnull().sum().sum()}")
print(f"\nDistribution quality :\n{df_wine['quality'].value_counts().sort_index()}")
df_wine.head(3)

HTTPError: HTTP Error 404: Not Found

In [ ]:
results_wine = run_pipeline_auto(wine_path, "quality", thread_id="wine")
print(f"Succes : {results_wine['success']} | Duree : {results_wine['duration']}s")

In [ ]:
transforms_wine = results_wine.get("transform_proposals", {}).get("transformations", [])
outliers_wine = results_wine.get("outlier_proposals", {}).get("outlier_actions", [])
qs_wine = results_wine.get("quality_score", {})
ba_wine = results_wine.get("report", {}).get("before_after", {})

print(f"Domaine : {results_wine.get('domain_context', {}).get('domain', '?')}")
print(f"\nTransformations : {len(transforms_wine)}")
for t in transforms_wine:
    print(f"  [{t.get('source', 'llm')}] {t['column']} : {t['action']}")

# Choix de scaling
scaling_actions = [t for t in transforms_wine if "scaling" in t["action"] or "transform" in t["action"]]
print(f"\nScaling/transforms : {len(scaling_actions)}")
for t in scaling_actions:
    print(f"  {t['column']} : {t['action']}")

print(f"\nOutlier actions : {len(outliers_wine)}")
for o in outliers_wine:
    print(f"  [{o.get('source', 'llm')}] {o['column']} : {o['method']}/{o['action']}")
print(f"\nQuality score : {qs_wine.get('overall', '?')}/100")
print(f"Shape : {ba_wine.get('original_shape', '?')} -> {ba_wine.get('final_shape', '?')}")

## 10. Robustesse — Target invalide

Test de la gestion d'erreur : que se passe-t-il si la colonne target n'existe pas dans le dataset ?
Le pipeline doit echouer proprement (fail-fast) avec un message clair.

In [ ]:
results_bad_target = run_pipeline_auto(url_tips, "colonne_inexistante", thread_id="bad_target")

print(f"Succes : {results_bad_target['success']}")
if not results_bad_target["success"]:
    print(f"Erreur capturee : {results_bad_target['errors'][0]}")
    print("\n-> Comportement attendu : fail-fast avec ValueError")
else:
    print("REGRESSION : le pipeline accepte une target inexistante")

## 11. Synthese

### 11.1 Tableau comparatif

In [ ]:
all_results = {
    "Titanic": results_titanic,
    "Tips": results_tips,
    "Diamonds": results_diamonds,
    "Penguins": results_penguins,
    "Extreme": results_extreme,
    "Adult": results_adult,
    "Mpg": results_mpg,
    "Wine": results_wine,
    "Bad Target": results_bad_target,
}

summary_rows = []
for name, r in all_results.items():
    n_transforms = len(r.get("transform_proposals", {}).get("transformations", []))
    n_outliers = len(r.get("outlier_proposals", {}).get("outlier_actions", []))
    qs = r.get("quality_score", {})
    ba = r.get("report", {}).get("before_after", {})
    conf_map = r.get("confidence_map", [])
    rule_ct = sum(1 for c in conf_map if c.get("source") == "rule") if conf_map else 0
    total_ct = len(conf_map) if conf_map else 0

    summary_rows.append({
        "Dataset": name,
        "Succes": r["success"],
        "Duree (s)": r["duration"],
        "Domaine": r.get("domain_context", {}).get("domain", "-"),
        "Transforms": n_transforms if r["success"] else "-",
        "Outlier actions": n_outliers if r["success"] else "-",
        "Quality /100": qs.get("overall", "-") if r["success"] else "-",
        "Deterministic %": f"{rule_ct/total_ct*100:.0f}%" if total_ct > 0 else "-",
        "Shape avant": ba.get("original_shape", "-"),
        "Shape apres": ba.get("final_shape", "-"),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

### 11.2 Validation des invariants

In [ ]:
print("VALIDATION DES INVARIANTS")
print("=" * 50)
all_pass = True

# 1. Target invalide -> fail-fast
ok = not results_bad_target["success"]
print(f"\n[{'OK' if ok else 'FAIL'}] Target invalide -> fail-fast")
all_pass &= ok

# 2. Colonnes triviales auto-drop
if results_extreme["success"]:
    ext_transforms = results_extreme.get("transform_proposals", {}).get("transformations", [])
    for col_name in ["all_nan_col", "constant_col", "high_cardinality_id"]:
        match = any(t["column"] == col_name and t["action"] == "drop_column" for t in ext_transforms)
        print(f"[{'OK' if match else 'FAIL'}] Auto-drop {col_name}")
        all_pass &= match

# 3. Validation Pydantic — aucune action inventee
valid_actions = {
    "impute_mean", "impute_median", "impute_mode", "drop_column", "drop_rows",
    "one_hot_encoding", "label_encoding", "ordinal_encoding", "frequency_encoding",
    "standard_scaling", "minmax_scaling", "robust_scaling",
    "log_transform", "sqrt_transform", "extract_datetime",
    "replace_sentinel", "cast_numeric",
}
all_actions = []
for name, r in all_results.items():
    if r["success"]:
        for t in r.get("transform_proposals", {}).get("transformations", []):
            all_actions.append((name, t["column"], t["action"]))
invalid = [(n, c, a) for n, c, a in all_actions if a not in valid_actions]
ok = len(invalid) == 0
print(f"[{'OK' if ok else 'FAIL'}] Pydantic — {len(all_actions)} actions, {len(invalid)} invalides")
if invalid:
    for n, c, a in invalid:
        print(f"  INVALIDE : {n}/{c} -> {a}")
all_pass &= ok

# 4. Multi-classe correctement detecte
meta = results_penguins.get("final_state", {}).get("target_meta", {})
ok = meta.get("task_type") == "classification" and meta.get("n_classes", 0) >= 3
print(f"[{'OK' if ok else 'FAIL'}] Penguins : multi-classe detecte (task_type={meta.get('task_type')}, n_classes={meta.get('n_classes')})")
all_pass &= ok

# 5. Tous les pipelines reussis (sauf bad_target)
for name, r in all_results.items():
    if name == "Bad Target":
        continue
    ok = r["success"]
    print(f"[{'OK' if ok else 'FAIL'}] Pipeline {name} termine avec succes")
    all_pass &= ok

# 6. Quality scores raisonnables (>= 50)
for name, r in all_results.items():
    if not r["success"]:
        continue
    score = r.get("quality_score", {}).get("overall", 0)
    ok = score >= 50
    print(f"[{'OK' if ok else 'FAIL'}] Quality score {name} : {score}/100 (>= 50)")
    all_pass &= ok

print(f"\n{'=' * 50}")
print(f"RESULTAT : {'TOUS LES TESTS PASSENT' if all_pass else 'CERTAINS TESTS ECHOUENT'}")

### 11.3 Analyse du ratio deterministe

In [ ]:
print("Ratio de decisions deterministes par dataset :")
print("-" * 50)

ratio_rows = []
for name, r in all_results.items():
    if not r["success"]:
        continue
    cm = r.get("confidence_map", [])
    if not cm:
        continue
    total = len(cm)
    rule_ct = sum(1 for c in cm if c.get("source") == "rule")
    high_ct = sum(1 for c in cm if c.get("confidence") == "high")
    low_ct = sum(1 for c in cm if c.get("confidence") == "low")
    ratio_rows.append({
        "Dataset": name,
        "Total decisions": total,
        "Rule": rule_ct,
        "LLM": total - rule_ct,
        "Deterministic %": f"{rule_ct/total*100:.0f}%",
        "High confidence": high_ct,
        "Low confidence": low_ct,
    })

pd.DataFrame(ratio_rows)

### 11.4 Rapport JSON complet (Titanic)

In [ ]:
report_titanic = results_titanic.get("report", {})
print(json.dumps(report_titanic, indent=2, default=str))

### 11.5 Limites connues et pistes d'amelioration

**Limites corrigees :**

| Limite | Correction |
|--------|------------|
| Target invalide non detectee | `_validate_target()` — fail-fast ValueError |
| Colonnes 100% NaN envoyees au LLM | `_detect_trivial_columns()` — auto-drop deterministe |
| Colonnes constantes ignorees | Detection dans `_detect_trivial_columns()` |
| Haute cardinalite -> OHE explosif | `frequency_encoding` pour >50 uniques |
| LLM invente des actions | Pydantic Literal + `validate_llm_response` avec retry |
| Actions inconnues skipees silencieusement | `raise ValueError` |
| Scalers non persistes | `joblib.dump` dans `data/outputs/artifacts/` |
| `describe()` trop volumineux | `_build_col_summary()` compact |
| Leakage non detecte | `_detect_leakage()` avec seuil 0.95 + interrupt force |
| Multicollinearite ignoree | `_detect_high_correlations()` (Pearson + Cramer's V) |
| Outlier remove sur test | Clip automatique au lieu de remove sur test |
| Pas de baseline evaluation | `_compute_baseline_score()` (dummy vs simple model) |

**Pistes d'amelioration :**

| Piste | Priorite |
|-------|----------|
| Seuils en config | Moyenne — deplacer les thresholds vers `model_config.yaml` |
| Appels LLM paralleles | Moyenne — paralleliser domain + diagnosis |
| Cross-validation | Basse — evaluer avec CV au lieu d'un simple split |
| Multi-provider LLM | Basse — support Anthropic, Mistral, modeles locaux |
| Sampling gros volumes | Basse — echantillonner avant analyse pour >50k lignes |